# Notebook 15 — LLM Integration

## Leadership and Management Book Recommendation System

### Objective

This notebook integrates a **Large Language Model (LLM)** with the validated content-based book recommendation system.

The LLM is not used to replace the recommendation algorithm.

Instead, it provides a conversational layer that can:

- interpret natural-language user requests;
- convert user interests into structured recommendation queries;
- present retrieved books in a readable format;
- explain why retrieved books may be relevant to the user's request;
- answer questions using metadata available in the project dataset.

The underlying recommendation engine remains grounded in the validated book catalog and similarity model.

---

## System Architecture

The intended architecture is:

**User Request**

↓  

**LLM Query Interpretation**

↓

**Validated Book Catalog / Recommendation Engine**

↓

**Enriched TF-IDF + Cosine Similarity**

↓

**Ranked Top-N Books**

↓

**LLM Grounded Explanation**

↓

**User-Facing Recommendation**

The LLM therefore operates around the recommendation engine rather than replacing it.

---

## Grounding Principle

The LLM must distinguish between:

1. information retrieved from the project's book dataset; and
2. general language-model knowledge.

For recommendation output, book-specific claims should be grounded in the metadata retrieved from the project.

The LLM should not invent:

- book titles;
- authors;
- ratings;
- descriptions;
- publication information;
- prices;
- recommendation scores;
- other unavailable metadata.

If information is unavailable in the project data, the system should identify it as unavailable rather than fabricate a value.

---

## Role of the LLM

The LLM layer will support two primary functions.

### 1. Query Interpretation

Example user request:

> I recently became a manager and want books about building trust, emotional intelligence, and leading a new team.

The LLM can extract concepts such as:

- new manager;
- trust;
- emotional intelligence;
- team leadership.

These concepts can then be used to interact with the recommendation pipeline.

### 2. Grounded Recommendation Explanation

After the recommendation engine retrieves books, the LLM can receive only the relevant retrieved metadata and generate a concise explanation of why each result relates to the user's request.

The recommendation ranking itself remains determined by the validated recommendation system.

---

## Relationship to Previous Models

The validated recommendation architecture remains:

**Enriched TF-IDF → Cosine Similarity → Duplicate Suppression → Ranked Top-N Recommendations**

The 32-dimensional autoencoder representation developed in Notebook 14 is retained as an experimental neural-network artifact but is not used as the primary recommendation representation because its semantic-preservation evaluation did not justify replacing the existing model.

---

## Notebook Workflow

This notebook will:

1. validate the required recommendation artifacts;
2. construct a grounded book-context representation;
3. define natural-language query handling;
4. create structured prompts for the LLM;
5. enforce dataset-grounded recommendation rules;
6. integrate the LLM interface;
7. test representative leadership and management queries;
8. evaluate grounding and response consistency;
9. save reusable components for the application layer.

The initial implementation will separate the recommendation logic from the external LLM API so that the grounding pipeline can be validated independently before API calls are introduced.

In [1]:
# ============================================================
# NOTEBOOK 15 — LLM INTEGRATION
# ENVIRONMENT AND ARTIFACT DISCOVERY
# ============================================================

from pathlib import Path
import sys
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"


# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

print("LLM INTEGRATION — ENVIRONMENT CHECK")
print("=" * 80)

print("Python version:")
print(sys.version)

print(
    "\nProject root exists:",
    PROJECT_ROOT.exists()
)

print(
    "Processed directory exists:",
    PROCESSED_DIR.exists()
)

print(
    "Models directory exists:",
    MODELS_DIR.exists()
)


# ------------------------------------------------------------
# Discover recommendation-related artifacts
# ------------------------------------------------------------

print("\nRECOMMENDATION / NLP ARTIFACTS")
print("=" * 80)

artifact_patterns = [
    "*recommend*",
    "*nlp*",
    "*tfidf*",
    "*cluster*"
]

found_artifacts = set()

for directory in [
    PROCESSED_DIR,
    MODELS_DIR
]:

    for pattern in artifact_patterns:

        for path in directory.glob(pattern):

            found_artifacts.add(path)


for path in sorted(found_artifacts):

    print(
        path.relative_to(PROJECT_ROOT)
    )


print(
    "\nArtifacts discovered:",
    len(found_artifacts)
)

LLM INTEGRATION — ENVIRONMENT CHECK
Python version:
3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:21:08) [Clang 20.1.8 ]

Project root exists: True
Processed directory exists: True
Models directory exists: True

RECOMMENDATION / NLP ARTIFACTS
data/processed/books_nlp_features.csv
data/processed/books_with_final_topic_clusters.csv
data/processed/core_vs_enriched_recommendations.csv
data/processed/final_cluster_representative_books.csv
data/processed/final_topic_cluster_summary.csv
data/processed/nlp_book_index.csv
data/processed/nlp_summary.csv
data/processed/recommendation_similarity_by_rank.csv
data/processed/recommendation_similarity_evaluation.csv
data/processed/recommendation_source_exposure.csv
data/processed/recommendation_topic_diversity.csv
models/core_tfidf_matrix.npz
models/core_tfidf_vectorizer.joblib
models/enriched_tfidf_matrix.npz
models/enriched_tfidf_vectorizer.joblib

Artifacts discovered: 15


## 1. Load the Validated Recommendation Representation

The LLM integration layer reuses the validated recommendation representation developed in the NLP and recommendation notebooks.

The primary recommendation artifacts are:

- `enriched_tfidf_matrix.npz`
- `enriched_tfidf_vectorizer.joblib`
- `nlp_book_index.csv`
- `books_nlp_features.csv`

The enriched TF-IDF representation was constructed from:

**Title + Authors + Subjects + Description**

and remains the primary numerical representation for recommendation.

### Custom Analyzer Compatibility

The fitted TF-IDF vectorizer uses the same custom boundary-aware analyzer developed during NLP preprocessing.

The analyzer is recreated exactly before loading the saved vectorizer so that the serialized model can be restored consistently.

The analyzer:

- preserves Unicode text;
- removes standard English stopwords;
- creates unigrams and within-field bigrams;
- prevents bigrams from crossing metadata-field boundaries.

No new vocabulary is fitted in this notebook.

The previously validated TF-IDF representation is loaded unchanged.

In [2]:
# ============================================================
# LOAD VALIDATED TF-IDF RECOMMENDATION ARTIFACTS
# ============================================================

from scipy.sparse import load_npz
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import joblib


# ------------------------------------------------------------
# Restore exact NLP analyzer used during fitting
# ------------------------------------------------------------

STOP_WORDS = set(
    ENGLISH_STOP_WORDS
)

FIELD_BOUNDARY = "zzfieldboundaryzz"


def boundary_aware_analyzer(document):

    features = []

    fields = document.split(
        FIELD_BOUNDARY
    )

    for field in fields:

        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        features.extend(
            tokens
        )

        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(
                    len(tokens) - 1
                )
            ]
        )

    return features


# ------------------------------------------------------------
# Artifact paths
# ------------------------------------------------------------

ENRICHED_MATRIX_PATH = (
    MODELS_DIR
    / "enriched_tfidf_matrix.npz"
)

ENRICHED_VECTORIZER_PATH = (
    MODELS_DIR
    / "enriched_tfidf_vectorizer.joblib"
)

NLP_INDEX_PATH = (
    PROCESSED_DIR
    / "nlp_book_index.csv"
)

NLP_FEATURES_PATH = (
    PROCESSED_DIR
    / "books_nlp_features.csv"
)


# ------------------------------------------------------------
# Load artifacts
# ------------------------------------------------------------

enriched_matrix = load_npz(
    ENRICHED_MATRIX_PATH
)

enriched_vectorizer = joblib.load(
    ENRICHED_VECTORIZER_PATH
)

nlp_index = pd.read_csv(
    NLP_INDEX_PATH
)

books_nlp = pd.read_csv(
    NLP_FEATURES_PATH
)


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("VALIDATED RECOMMENDATION ARTIFACTS")
print("=" * 80)

print(
    "Enriched TF-IDF shape:",
    enriched_matrix.shape
)

print(
    "NLP index rows:",
    len(nlp_index)
)

print(
    "NLP feature rows:",
    len(books_nlp)
)

print(
    "\nMatrix/index aligned:",
    enriched_matrix.shape[0]
    == len(nlp_index)
)

print(
    "Expected TF-IDF dimensions:",
    enriched_matrix.shape[1]
    == 5130
)

print(
    "Index book IDs unique:",
    nlp_index["book_id"].is_unique
)

print(
    "Feature book IDs unique:",
    books_nlp["book_id"].is_unique
)

print(
    "\nVectorizer vocabulary size:",
    len(
        enriched_vectorizer.vocabulary_
    )
)

print(
    "Vocabulary matches matrix:",
    len(enriched_vectorizer.vocabulary_)
    == enriched_matrix.shape[1]
)

print(
    "\nZero TF-IDF vectors:",
    int(
        (
            enriched_matrix.getnnz(axis=1)
            == 0
        ).sum()
    )
)

VALIDATED RECOMMENDATION ARTIFACTS
Enriched TF-IDF shape: (2067, 5130)
NLP index rows: 2067
NLP feature rows: 2067

Matrix/index aligned: True
Expected TF-IDF dimensions: True
Index book IDs unique: True
Feature book IDs unique: True

Vectorizer vocabulary size: 5130
Vocabulary matches matrix: True

Zero TF-IDF vectors: 27


## 2. Inspect Available Metadata for LLM Grounding

The LLM should only receive book-specific information that is available from validated project artifacts.

Before constructing the grounding records, the available columns in the NLP feature table and NLP index are inspected.

This prevents the LLM integration layer from assuming that a metadata field exists when it is not actually available.

The inspection will identify which fields can safely support:

- book identification;
- author information;
- subjects and topics;
- descriptions;
- source provenance;
- recommendation explanations;
- other contextual metadata.

The grounding schema will be constructed only after the available fields have been confirmed.

In [3]:
# ============================================================
# INSPECT AVAILABLE LLM GROUNDING METADATA
# ============================================================

print("NLP BOOK INDEX")
print("=" * 80)

print(
    "Shape:",
    nlp_index.shape
)

print("\nColumns:")

for column in nlp_index.columns:
    print(
        f"- {column}"
    )


print("\n" + "=" * 80)
print("NLP FEATURE TABLE")
print("=" * 80)

print(
    "Shape:",
    books_nlp.shape
)

print("\nColumns:")

for column in books_nlp.columns:
    print(
        f"- {column}"
    )


# ------------------------------------------------------------
# Shared columns
# ------------------------------------------------------------

shared_columns = sorted(
    set(nlp_index.columns)
    & set(books_nlp.columns)
)

print("\n" + "=" * 80)
print("SHARED COLUMNS")
print("=" * 80)

for column in shared_columns:
    print(
        f"- {column}"
    )


# ------------------------------------------------------------
# Inspect one representative record
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE NLP INDEX RECORD")
print("=" * 80)

sample_index = (
    nlp_index.iloc[0]
)

for column, value in sample_index.items():

    print(
        f"{column}: {value}"
    )


print("\n" + "=" * 80)
print("SAMPLE NLP FEATURE RECORD")
print("=" * 80)

sample_features = (
    books_nlp.iloc[0]
)

for column, value in sample_features.items():

    text = str(value)

    if len(text) > 300:
        text = (
            text[:300]
            + "..."
        )

    print(
        f"{column}: {text}"
    )

NLP BOOK INDEX
Shape: (2067, 6)

Columns:
- matrix_row
- book_id
- canonical_title
- source_group
- core_zero_vector
- enriched_zero_vector

NLP FEATURE TABLE
Shape: (2067, 9)

Columns:
- book_id
- canonical_title
- source_group
- core_text_boundary
- enriched_text_boundary
- core_active_features_final
- enriched_active_features_final
- core_zero_vector_final
- enriched_zero_vector_final

SHARED COLUMNS
- book_id
- canonical_title
- source_group

SAMPLE NLP INDEX RECORD
matrix_row: 0
book_id: BOOK00001
canonical_title: Principle-Centered Leadership
source_group: Open Library only
core_zero_vector: False
enriched_zero_vector: False

SAMPLE NLP FEATURE RECORD
book_id: BOOK00001
canonical_title: Principle-Centered Leadership
source_group: Open Library only
core_text_boundary: principle-centered leadership zzfieldboundaryzz stephen r covey
enriched_text_boundary: principle-centered leadership zzfieldboundaryzz stephen r covey zzfieldboundaryzz leadership psychological aspects of success su

## 3. Locate Structured Book Metadata

The NLP feature table contains the text representation used for modelling, but it does not provide separate human-readable fields for authors, subjects, descriptions, publication metadata, and other book attributes.

The combined `enriched_text_boundary` field should not be reverse-engineered into structured metadata because the project already maintains structured book records from the earlier data-integration stage.

For LLM grounding, the system should use:

- the validated TF-IDF representation for recommendation retrieval; and
- the original structured metadata for recommendation explanations.

This separation preserves the role of each artifact:

**TF-IDF matrix → numerical retrieval**

**Structured book metadata → grounded LLM context**

The next step therefore identifies the integrated book dataset and inspects its available fields before defining the final grounding schema.

In [4]:
# ============================================================
# LOCATE STRUCTURED BOOK METADATA
# ============================================================

print("STRUCTURED BOOK DATA DISCOVERY")
print("=" * 80)


# Search project data folders for likely book-level datasets
candidate_files = []

for directory in [
    DATA_DIR,
    PROCESSED_DIR,
    DATA_DIR / "final"
]:

    if directory.exists():

        for path in directory.glob("*.csv"):

            filename = path.name.lower()

            if any(
                keyword in filename
                for keyword in [
                    "master",
                    "book",
                    "integrat"
                ]
            ):

                candidate_files.append(
                    path
                )


# Remove duplicates
candidate_files = sorted(
    set(candidate_files)
)


for path in candidate_files:

    try:

        preview = pd.read_csv(
            path,
            nrows=3
        )

        print(
            "\n",
            path.relative_to(
                PROJECT_ROOT
            )
        )

        print(
            "Columns:",
            list(preview.columns)
        )

    except Exception as error:

        print(
            "\nCould not inspect:",
            path.relative_to(
                PROJECT_ROOT
            )
        )

        print(
            "Reason:",
            error
        )


print(
    "\nCandidate files found:",
    len(candidate_files)
)

STRUCTURED BOOK DATA DISCOVERY

 data/final/book_metadata_coverage.csv
Columns: ['field', 'available', 'coverage_pct']

 data/final/books_master.csv
Columns: ['book_id', 'canonical_title', 'authors', 'description', 'subjects', 'first_publish_year', 'average_rating', 'ratings_count', 'edition_count', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_url', 'openlibrary_key', 'source_openlibrary', 'source_leadershipnow', 'publication_year_observed', 'title_normalized']

 data/final/integration_quality_summary.csv
Columns: ['metric', 'value']

 data/final/openlibrary_integrated.csv
Columns: ['openlibrary_key', 'title', 'authors', 'author_keys', 'first_publish_year', 'publish_dates', 'publishers', 'isbn_10', 'isbn_13', 'all_isbns', 'languages', 'subjects', 'edition_count', 'ratings_average', 'ratings_count', 'ratings_count_1', 'ratings_count_2', 'ratings_count_3', 'ratings_count_4', 'ratings_count_5', 'want_to_read_count', 'currently_reading_count', 'already_read

## 4. Construct the LLM Grounding Catalog

The LLM requires a structured source of book information that is separate from the numerical TF-IDF representation.

The grounding catalog combines:

1. the authoritative TF-IDF row-to-book mapping from `nlp_book_index.csv`; and
2. structured book metadata and topic-cluster information from `books_with_final_topic_clusters.csv`.

A **left join** is performed from the NLP index so that all 2,067 books in the recommendation universe are preserved.

### Grounding Fields

The initial grounding catalog retains:

- `matrix_row`
- `book_id`
- `canonical_title`
- `authors`
- `description`
- `subjects`
- `first_publish_year`
- `publication_year_observed`
- `average_rating`
- `ratings_count`
- `edition_count`
- `want_to_read_count`
- `currently_reading_count`
- `already_read_count`
- `cover_url`
- `source_group`
- `topic_cluster`
- `cluster_label`
- `enriched_zero_vector`

These fields serve different purposes:

- **TF-IDF retrieval:** `matrix_row`
- **Book identification:** title and authors
- **Semantic explanation:** subjects and description
- **Context:** publication information and topic cluster
- **Descriptive metadata:** ratings and reader-engagement fields
- **Application interface:** cover URL and source group

### Missing-Value Policy

Missing metadata is retained as missing.

The LLM layer must not infer or fabricate unavailable:

- authors;
- descriptions;
- subjects;
- publication years;
- ratings;
- engagement statistics;
- cluster assignments.

A missing cluster label means that the book was not assigned to the final topic-clustering population; it does not mean that the book has no topic.

In [5]:
# ============================================================
# CONSTRUCT LLM GROUNDING CATALOG
# ============================================================

CLUSTER_BOOKS_PATH = (
    PROCESSED_DIR
    / "books_with_final_topic_clusters.csv"
)

cluster_books = pd.read_csv(
    CLUSTER_BOOKS_PATH
)


# ------------------------------------------------------------
# Select only fields required for grounding
# ------------------------------------------------------------

grounding_columns = [
    "book_id",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_url",
    "topic_cluster",
    "cluster_label"
]

structured_grounding = (
    cluster_books[
        grounding_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Preserve authoritative NLP index order
# ------------------------------------------------------------

grounding_catalog = (
    nlp_index
    .merge(
        structured_grounding,
        on="book_id",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Arrange final grounding schema
# ------------------------------------------------------------

grounding_catalog = grounding_catalog[
    [
        "matrix_row",
        "book_id",
        "canonical_title",
        "authors",
        "description",
        "subjects",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "edition_count",
        "want_to_read_count",
        "currently_reading_count",
        "already_read_count",
        "cover_url",
        "source_group",
        "topic_cluster",
        "cluster_label",
        "enriched_zero_vector"
    ]
]


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("LLM GROUNDING CATALOG")
print("=" * 80)

print(
    "Rows:",
    len(grounding_catalog)
)

print(
    "Columns:",
    grounding_catalog.shape[1]
)

print(
    "Unique book IDs:",
    grounding_catalog["book_id"].nunique()
)

print(
    "Matrix rows unique:",
    grounding_catalog["matrix_row"].is_unique
)

print(
    "Matrix order preserved:",
    np.array_equal(
        grounding_catalog["matrix_row"].to_numpy(),
        np.arange(len(grounding_catalog))
    )
)

print(
    "TF-IDF rows aligned:",
    len(grounding_catalog)
    == enriched_matrix.shape[0]
)


# ------------------------------------------------------------
# Metadata coverage
# ------------------------------------------------------------

print("\nGROUNDING METADATA COVERAGE")
print("=" * 80)

coverage_columns = [
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "cover_url",
    "topic_cluster",
    "cluster_label"
]

for column in coverage_columns:

    available = (
        grounding_catalog[column]
        .notna()
        .sum()
    )

    percentage = (
        available
        / len(grounding_catalog)
        * 100
    )

    print(
        f"{column:<28}"
        f"{available:>5} "
        f"({percentage:>6.2f}%)"
    )


# ------------------------------------------------------------
# Cluster coverage
# ------------------------------------------------------------

clustered_books = (
    grounding_catalog[
        "topic_cluster"
    ]
    .notna()
    .sum()
)

print("\nCLUSTER COVERAGE")
print("=" * 80)

print(
    "Books with cluster metadata:",
    clustered_books
)

print(
    "Books without cluster metadata:",
    len(grounding_catalog)
    - clustered_books
)


# ------------------------------------------------------------
# Zero-vector integrity
# ------------------------------------------------------------

print("\nTF-IDF INTEGRITY")
print("=" * 80)

print(
    "Zero-vector books:",
    grounding_catalog[
        "enriched_zero_vector"
    ].sum()
)

print(
    "Eligible recommendation books:",
    (
        ~grounding_catalog[
            "enriched_zero_vector"
        ]
    ).sum()
)

LLM GROUNDING CATALOG
Rows: 2067
Columns: 19
Unique book IDs: 2067
Matrix rows unique: True
Matrix order preserved: True
TF-IDF rows aligned: True

GROUNDING METADATA COVERAGE
authors                      1884 ( 91.15%)
description                   192 (  9.29%)
subjects                     1884 ( 91.15%)
first_publish_year            947 ( 45.82%)
publication_year_observed     934 ( 45.19%)
average_rating                282 ( 13.64%)
ratings_count                 282 ( 13.64%)
edition_count                 950 ( 45.96%)
cover_url                    1705 ( 82.49%)
topic_cluster                1884 ( 91.15%)
cluster_label                1884 ( 91.15%)

CLUSTER COVERAGE
Books with cluster metadata: 1884
Books without cluster metadata: 183

TF-IDF INTEGRITY
Zero-vector books: 27
Eligible recommendation books: 2040


## 5. Natural-Language Semantic Retrieval

The recommendation system developed in Notebook 12 primarily retrieves books based on similarity to an existing book.

For the conversational application, users should also be able to describe their needs in natural language.

Example:

> "I am a new manager and want to improve team motivation, communication, and conflict management."

The natural-language request is transformed using the **same fitted enriched TF-IDF vectorizer** used by the validated recommendation system.

Cosine similarity is then calculated between the query vector and the 2,067 book vectors.

Only books with valid non-zero enriched TF-IDF representations are eligible for retrieval.

### Important Methodological Distinction

This step does not train a new recommendation model.

It applies the previously fitted TF-IDF vocabulary and weighting scheme to a new user query.

The workflow is:

**Natural-Language Query**

→ **Existing Enriched TF-IDF Vectorizer**

→ **Query TF-IDF Vector**

→ **Cosine Similarity Against Book Matrix**

→ **Ranked Top-N Books**

The LLM will later operate on these retrieved books rather than generating book recommendations independently.

This creates a retrieval-grounded architecture in which the dataset determines which books are recommended.

In [6]:
# ============================================================
# NATURAL-LANGUAGE BOOK RETRIEVAL
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def retrieve_books_from_query(
    query,
    top_n=10
):
    """
    Retrieve the most similar books for a natural-language
    user query using the validated enriched TF-IDF model.
    """

    # --------------------------------------------------------
    # Validate query
    # --------------------------------------------------------

    if not isinstance(query, str):
        return pd.DataFrame()

    query = query.strip()

    if not query:
        return pd.DataFrame()


    # --------------------------------------------------------
    # Transform query using EXISTING fitted vectorizer
    # --------------------------------------------------------

    query_vector = (
        enriched_vectorizer.transform(
            [query]
        )
    )


    # Query may contain no recognized vocabulary
    if query_vector.nnz == 0:

        print(
            "The query contains no vocabulary recognized "
            "by the fitted TF-IDF model."
        )

        return pd.DataFrame()


    # --------------------------------------------------------
    # Calculate cosine similarity
    # --------------------------------------------------------

    similarities = cosine_similarity(
        query_vector,
        enriched_matrix
    ).ravel()


    # --------------------------------------------------------
    # Exclude invalid book vectors
    # --------------------------------------------------------

    valid_mask = (
        ~grounding_catalog[
            "enriched_zero_vector"
        ].astype(bool).to_numpy()
    )

    similarities[
        ~valid_mask
    ] = -1.0


    # --------------------------------------------------------
    # Retain positive similarities only
    # --------------------------------------------------------

    candidate_indices = np.where(
        similarities > 0
    )[0]

    if len(candidate_indices) == 0:

        print(
            "No positively similar books were found."
        )

        return pd.DataFrame()


    # --------------------------------------------------------
    # Rank candidates
    # --------------------------------------------------------

    ranked_indices = candidate_indices[
        np.argsort(
            similarities[
                candidate_indices
            ]
        )[::-1]
    ]


    ranked_indices = ranked_indices[
        :top_n
    ]


    # --------------------------------------------------------
    # Retrieve grounding metadata
    # --------------------------------------------------------

    results = (
        grounding_catalog
        .iloc[ranked_indices]
        .copy()
    )

    results.insert(
        0,
        "recommendation_rank",
        np.arange(
            1,
            len(results) + 1
        )
    )

    results.insert(
        3,
        "similarity_score",
        similarities[
            ranked_indices
        ]
    )


    # --------------------------------------------------------
    # Return useful fields
    # --------------------------------------------------------

    result_columns = [
        "recommendation_rank",
        "book_id",
        "canonical_title",
        "similarity_score",
        "authors",
        "subjects",
        "description",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "source_group",
        "topic_cluster",
        "cluster_label",
        "cover_url"
    ]

    return (
        results[
            result_columns
        ]
        .reset_index(
            drop=True
        )
    )

In [7]:
# ============================================================
# TEST NATURAL-LANGUAGE RETRIEVAL
# ============================================================

test_query = (
    "I am a new manager and want to improve "
    "team motivation, communication, emotional intelligence, "
    "trust, and conflict management."
)

query_results = retrieve_books_from_query(
    query=test_query,
    top_n=10
)


print("USER QUERY")
print("=" * 80)
print(test_query)

print("\nTOP-10 RETRIEVED BOOKS")
print("=" * 80)


display_columns = [
    "recommendation_rank",
    "canonical_title",
    "authors",
    "similarity_score",
    "cluster_label"
]


if not query_results.empty:

    display(
        query_results[
            display_columns
        ].style.format({
            "similarity_score": "{:.4f}"
        })
    )

    print(
        "\nRecommendations returned:",
        len(query_results)
    )

    print(
        "Unique book IDs:",
        query_results[
            "book_id"
        ].nunique()
    )

    print(
        "Similarity range:",
        f"{query_results['similarity_score'].min():.4f}",
        "to",
        f"{query_results['similarity_score'].max():.4f}"
    )

else:

    print(
        "No recommendations returned."
    )

USER QUERY
I am a new manager and want to improve team motivation, communication, emotional intelligence, trust, and conflict management.

TOP-10 RETRIEVED BOOKS


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,Emotional Intelligence : How to Improve Emotional Intelligence,['Kati Haylock'],0.4874,Emotional Intelligence
1,2,Emotional Intelligence,['Tanvir Shakil'],0.4087,Emotional Intelligence
2,3,Emotional Intelligence Mastery 2. 0,['Emotional Intelligence Academy'],0.3637,Emotional Intelligence
3,4,Emotional Intelligence : 3 Books in 1,['Emotional Pathway'],0.3480,Emotional Intelligence
4,5,Communicate with emotional intelligence,"['John Eaton', 'Roy Johnson']",0.3405,Emotional Intelligence
5,6,Emotional intelligence,['Jill Dann'],0.3202,Emotional Intelligence
6,7,If You Want Something Done,['Nikki R. Haley'],0.3102,Future Thinking and Personal Development
7,8,How to Be a Super-Effective Manager,['Clive T. Goodworth'],0.3066,Broad Leadership and Success
8,9,Emotional Intelligence for Dummies,['Steven J. Stein'],0.2898,Emotional Intelligence
9,10,Emotional Intelligence,['James W. Williams'],0.2887,Emotional Intelligence



Recommendations returned: 10
Unique book IDs: 10
Similarity range: 0.2887 to 0.4874


## 6. Evaluate Natural-Language Retrieval Across Leadership Needs

The first natural-language query successfully retrieved relevant books, but the results were heavily concentrated around the phrase **"emotional intelligence."**

This behavior is consistent with TF-IDF retrieval because exact or highly distinctive vocabulary can dominate cosine similarity.

A conversational recommendation system should therefore be evaluated across multiple leadership and management scenarios before the LLM layer is introduced.

The following tests examine whether the retrieval system responds meaningfully to different user intents:

1. new-manager and team leadership;
2. conflict and difficult conversations;
3. strategy and decision-making;
4. organizational change;
5. entrepreneurship and innovation;
6. employee motivation and engagement.

For each query, the evaluation records:

- number of recommendations returned;
- number of unique books;
- mean similarity;
- maximum similarity;
- number of unique topic clusters represented;
- dominant topic cluster;
- dominant-cluster concentration.

This is a diagnostic evaluation rather than a supervised accuracy test because no human relevance labels are available.

The objective is to determine whether natural-language retrieval produces differentiated results across different leadership needs and whether individual concepts excessively dominate the recommendation list.

In [8]:
# ============================================================
# MULTI-QUERY NATURAL-LANGUAGE RETRIEVAL EVALUATION
# ============================================================

diagnostic_queries = {
    
    "New Manager": (
        "I recently became a manager and need help leading "
        "a team, building trust, communicating effectively, "
        "and motivating employees."
    ),

    "Conflict Management": (
        "I need to manage workplace conflict, handle difficult "
        "conversations, negotiate disagreements, and improve "
        "communication between team members."
    ),

    "Strategy": (
        "I want to improve strategic thinking, business "
        "decision making, competitive strategy, and long-term "
        "planning."
    ),

    "Organizational Change": (
        "I am leading organizational change and need help "
        "managing transformation, employee resistance, culture, "
        "and change leadership."
    ),

    "Entrepreneurship": (
        "I want to build a startup, develop an entrepreneurial "
        "mindset, innovate, identify opportunities, and grow "
        "a new business."
    ),

    "Employee Motivation": (
        "I want to motivate employees, improve engagement, "
        "develop high-performing teams, and strengthen "
        "workplace performance."
    )
}


diagnostic_results = {}
diagnostic_summary = []


for query_name, query_text in diagnostic_queries.items():

    results = retrieve_books_from_query(
        query=query_text,
        top_n=10
    )

    diagnostic_results[
        query_name
    ] = results

    if results.empty:

        diagnostic_summary.append({
            "query": query_name,
            "recommendations": 0,
            "unique_books": 0,
            "mean_similarity": np.nan,
            "max_similarity": np.nan,
            "unique_clusters": 0,
            "dominant_cluster": None,
            "dominant_cluster_count": 0,
            "dominant_cluster_pct": np.nan
        })

        continue


    cluster_counts = (
        results["cluster_label"]
        .fillna("No cluster")
        .value_counts()
    )

    dominant_cluster = (
        cluster_counts.index[0]
    )

    dominant_count = int(
        cluster_counts.iloc[0]
    )


    diagnostic_summary.append({
        "query": query_name,

        "recommendations":
            len(results),

        "unique_books":
            results["book_id"].nunique(),

        "mean_similarity":
            results["similarity_score"].mean(),

        "max_similarity":
            results["similarity_score"].max(),

        "unique_clusters":
            results["cluster_label"]
            .nunique(dropna=True),

        "dominant_cluster":
            dominant_cluster,

        "dominant_cluster_count":
            dominant_count,

        "dominant_cluster_pct":
            dominant_count
            / len(results)
            * 100
    })


diagnostic_summary_df = pd.DataFrame(
    diagnostic_summary
)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print(
    "NATURAL-LANGUAGE RETRIEVAL DIAGNOSTIC"
)

print("=" * 100)

display(
    diagnostic_summary_df.style.format({
        "mean_similarity": "{:.4f}",
        "max_similarity": "{:.4f}",
        "dominant_cluster_pct": "{:.1f}%"
    })
)


# ------------------------------------------------------------
# Show Top-5 titles for each query
# ------------------------------------------------------------

for query_name, results in diagnostic_results.items():

    print(
        "\n",
        "=" * 100,
        sep=""
    )

    print(
        query_name.upper()
    )

    print(
        "=" * 100
    )

    if results.empty:

        print(
            "No recommendations."
        )

        continue

    display(
        results[
            [
                "recommendation_rank",
                "canonical_title",
                "authors",
                "similarity_score",
                "cluster_label"
            ]
        ]
        .head(5)
        .style.format({
            "similarity_score": "{:.4f}"
        })
    )

NATURAL-LANGUAGE RETRIEVAL DIAGNOSTIC


,query,recommendations,unique_books,mean_similarity,max_similarity,unique_clusters,dominant_cluster,dominant_cluster_count,dominant_cluster_pct
0,New Manager,10,10,0.2392,0.4753,5,Organizational Culture and Trust,5,50.0%
1,Conflict Management,10,10,0.1944,0.3142,5,Organizational Culture and Trust,4,40.0%
2,Strategy,10,10,0.2819,0.3611,4,Decision Making,6,60.0%
3,Organizational Change,10,10,0.2390,0.3644,4,Change Management,7,70.0%
4,Entrepreneurship,10,10,0.2397,0.3185,3,Entrepreneurial and Growth Mindset,7,70.0%
5,Employee Motivation,10,10,0.2087,0.3527,3,Performance Management,4,40.0%



NEW MANAGER


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,Building Trust,['Darryl Stickel'],0.4753,Organizational Culture and Trust
1,2,How to Be a Super-Effective Manager,['Clive T. Goodworth'],0.2954,Broad Leadership and Success
2,3,The Trifecta of Trust,['Joseph R. Folkman'],0.2589,Organizational Culture and Trust
3,4,Trust,['Henry Cloud'],0.2472,Organizational Culture and Trust
4,5,Building Better Organizations,['Claudy Jules'],0.1990,Broad Leadership and Success



CONFLICT MANAGEMENT


...

,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,Enterprise Change Management,"['David Miller', 'Audra Proctor']",0.3644,Change Management
1,2,Leading Organizational Transformation,['Alejandro Reyes'],0.2989,Leading Self and Others
2,3,The Secret of Culture Change,"['Manoel Amorim, Jay B. Barney and Carlos Julio']",0.2659,Change Management
3,4,The complete idiot's guide to change management,['Jeffrey P. Davidson'],0.2252,Change Management
4,5,Leadership and change management,['Annabel C. Beerel'],0.2208,Change Management



ENTREPRENEURSHIP


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,The Entrepreneurial Mindset Advantage,['Gary G. Schoeniger'],0.3185,Entrepreneurial and Growth Mindset
1,2,If You Want Something Done,['Nikki R. Haley'],0.2864,Future Thinking and Personal Development
2,3,Grow,['Michael McFall'],0.2835,Entrepreneurial and Growth Mindset
3,4,Unstoppable Mindset,['Alden Mills'],0.2650,Entrepreneurial and Growth Mindset
4,5,The Sweaty Startup,['Nick Huber'],0.2412,Entrepreneurial and Growth Mindset



EMPLOYEE MOTIVATION


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,If You Want Something Done,['Nikki R. Haley'],0.3527,Future Thinking and Personal Development
1,2,The Brain-Friendly Workplace,['Friederike Fabritius'],0.2675,Leadership Development
2,3,Get What You Want,['Julie Solomon'],0.2358,Future Thinking and Personal Development
3,4,All the War They Want,['Jeffrey J. Engle'],0.2047,Future Thinking and Personal Development
4,5,Team leadership,"['Parker, Glenn M.']",0.1824,Leadership Development


## 7. Construct Grounded Context for the LLM

The natural-language retrieval evaluation demonstrates that the TF-IDF model responds to different leadership and management intents and generally retrieves semantically relevant books.

The next step is to convert retrieved recommendations into a controlled context that can be supplied to an LLM.

### Retrieval-Augmented Generation Principle

The LLM does not search the complete book catalog independently.

Instead:

1. the user's request is transformed using the validated TF-IDF model;
2. the recommendation system retrieves the most relevant books;
3. structured metadata for those books is converted into a grounding context;
4. the LLM receives the user request together with only the retrieved book evidence;
5. the LLM explains the recommendations without changing the retrieval ranking.

This follows a retrieval-augmented generation (RAG) pattern:

**User Query → Retrieval → Grounded Context → LLM Explanation**

### Grounding Rules

The context supplied to the LLM follows several constraints:

- only retrieved books may be recommended;
- recommendation rank is determined by the retrieval model;
- similarity scores are generated by the recommendation model, not the LLM;
- missing metadata remains explicitly unavailable;
- descriptions are included only when available;
- ratings are included only when available;
- topic-cluster labels may support explanation but do not determine ranking;
- the LLM must not invent additional book metadata.

This separation maintains a clear distinction between:

**retrieval evidence** and **natural-language generation**.

In [9]:
# ============================================================
# BUILD CONTROLLED LLM GROUNDING CONTEXT
# ============================================================

def clean_grounding_value(value):
    """
    Convert a metadata value into safe grounding text.
    Missing values are represented explicitly.
    """

    if pd.isna(value):
        return "Not available"

    text = str(value).strip()

    if not text:
        return "Not available"

    return text


def truncate_grounding_text(
    value,
    max_chars=600
):
    """
    Limit long metadata fields to control prompt size.
    """

    text = clean_grounding_value(value)

    if text == "Not available":
        return text

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars].rstrip()
        + "..."
    )


def build_llm_grounding_context(
    recommendations
):
    """
    Convert retrieved recommendations into structured,
    dataset-grounded text for an LLM.
    """

    if recommendations.empty:
        return "No retrieved books are available."

    context_blocks = []

    for _, row in recommendations.iterrows():

        block = [
            f"Recommendation Rank: "
            f"{int(row['recommendation_rank'])}",

            f"Book ID: "
            f"{clean_grounding_value(row['book_id'])}",

            f"Title: "
            f"{clean_grounding_value(row['canonical_title'])}",

            f"Authors: "
            f"{clean_grounding_value(row['authors'])}",

            f"Similarity Score: "
            f"{row['similarity_score']:.4f}",

            f"Subjects: "
            f"{truncate_grounding_text(row['subjects'], 500)}",

            f"Description: "
            f"{truncate_grounding_text(row['description'], 700)}",

            f"First Publication Year: "
            f"{clean_grounding_value(row['first_publish_year'])}",

            f"Observed Publication Year: "
            f"{clean_grounding_value(row['publication_year_observed'])}",

            f"Average Rating: "
            f"{clean_grounding_value(row['average_rating'])}",

            f"Ratings Count: "
            f"{clean_grounding_value(row['ratings_count'])}",

            f"Topic Cluster: "
            f"{clean_grounding_value(row['cluster_label'])}",

            f"Source Group: "
            f"{clean_grounding_value(row['source_group'])}"
        ]

        context_blocks.append(
            "\n".join(block)
        )

    return "\n\n---\n\n".join(
        context_blocks
    )

In [10]:
# ============================================================
# TEST LLM GROUNDING CONTEXT
# ============================================================

grounding_test_query = (
    diagnostic_queries[
        "Organizational Change"
    ]
)

grounding_test_results = (
    diagnostic_results[
        "Organizational Change"
    ].head(5)
)

grounding_context = (
    build_llm_grounding_context(
        grounding_test_results
    )
)


print("USER QUERY")
print("=" * 80)
print(
    grounding_test_query
)

print("\nLLM GROUNDING CONTEXT")
print("=" * 80)

print(
    grounding_context
)

print("\n" + "=" * 80)

print(
    "Books supplied to LLM:",
    len(grounding_test_results)
)

print(
    "Grounding context characters:",
    len(grounding_context)
)

USER QUERY
I am leading organizational change and need help managing transformation, employee resistance, culture, and change leadership.

LLM GROUNDING CONTEXT
Recommendation Rank: 1
Book ID: BOOK00465
Title: Enterprise Change Management
Authors: ['David Miller', 'Audra Proctor']
Similarity Score: 0.3644
Subjects: ['Organizational change', 'Leadership']
Description: Not available
First Publication Year: 2016.0
Observed Publication Year: Not available
Average Rating: Not available
Ratings Count: Not available
Topic Cluster: Change Management
Source Group: Open Library only

---

Recommendation Rank: 2
Book ID: BOOK01926
Title: Leading Organizational Transformation
Authors: ['Alejandro Reyes']
Similarity Score: 0.2989
Subjects: []
Description: Not available
First Publication Year: Not available
Observed Publication Year: 2025.0
Average Rating: Not available
Ratings Count: Not available
Topic Cluster: Leading Self and Others
Source Group: LeadershipNow only

---

Recommendation Rank: 3
B

## 8. Design the Grounded LLM Prompt

The retrieval system now provides a controlled set of book recommendations and structured metadata.

The next stage defines the instructions given to the Large Language Model.

The prompt is designed to prevent the LLM from functioning as an independent recommendation engine.

Instead, the LLM acts as a **grounded recommendation explainer**.

### LLM Responsibilities

The LLM may:

- interpret the user's leadership or management need;
- explain why each retrieved book may be relevant;
- summarize the available evidence;
- present recommendations conversationally;
- acknowledge missing metadata.

The LLM may not:

- introduce books that were not retrieved;
- change the recommendation ranking;
- invent authors, ratings, publication information, subjects, or descriptions;
- claim that a book covers a topic unless the supplied evidence reasonably supports that statement;
- treat the similarity score as a quality rating;
- describe a missing field as though it were known.

### Interpretation of Similarity Scores

The TF-IDF cosine similarity score represents textual similarity between the user's query and the available book metadata.

It does **not** represent:

- book quality;
- probability that the user will like the book;
- expert evaluation;
- predicted rating;
- popularity;
- recommendation accuracy.

### Grounded Generation

The final prompt contains three components:

1. **System instructions** defining the LLM's role and restrictions;
2. **User request** containing the original natural-language need;
3. **Retrieved evidence** containing only the books selected by the recommendation engine.

This architecture maintains separation between:

**Recommendation Retrieval**

and

**Natural-Language Explanation**

In [11]:
# ============================================================
# GROUNDED LLM PROMPT BUILDER
# ============================================================

LLM_SYSTEM_INSTRUCTIONS = """
You are the explanation layer of a leadership and management
book recommendation system.

The books have already been selected and ranked by a
TF-IDF cosine-similarity retrieval model.

Your role is to explain the retrieved recommendations clearly
and accurately.

STRICT GROUNDING RULES:

1. Recommend only books contained in the RETRIEVED BOOK EVIDENCE.
2. Preserve the recommendation ranking supplied by the retrieval model.
3. Do not introduce additional book titles.
4. Do not invent authors, descriptions, subjects, ratings,
   publication years, or other metadata.
5. If metadata says "Not available", do not infer the missing value.
6. Base book-specific explanations only on the supplied evidence.
7. A similarity score represents textual similarity to the user's
   request. It is not a quality score, predicted rating, probability
   of satisfaction, or expert evaluation.
8. Topic-cluster labels provide supplementary context only.
9. Do not claim that the retrieval model proves that a book is
   objectively the best book for the user.
10. Clearly acknowledge when the available metadata is limited.

RESPONSE FORMAT:

Begin with a short interpretation of the user's need.

Then present the recommendations in the exact supplied ranking.

For each book provide:
- rank and title;
- author, when available;
- a concise explanation of relevance based only on supplied evidence;
- similarity score labelled "Retrieval similarity";
- topic cluster, when available.

Finish with a short note explaining that the recommendations are
generated from textual similarity within the project's book catalog.
""".strip()


def build_grounded_llm_prompt(
    user_query,
    recommendations
):
    """
    Construct the complete grounded prompt for the LLM.
    """

    grounding_context = (
        build_llm_grounding_context(
            recommendations
        )
    )

    user_prompt = f"""
USER REQUEST
============

{user_query}


RETRIEVED BOOK EVIDENCE
=======================

{grounding_context}


TASK
====

Explain these retrieved recommendations to the user.

Use only the supplied book evidence.

Preserve the supplied ranking.

Do not add books or unsupported book-specific claims.
""".strip()

    return {
        "system": LLM_SYSTEM_INSTRUCTIONS,
        "user": user_prompt
    }

In [12]:
# ============================================================
# VALIDATE GROUNDED LLM PROMPT
# ============================================================

test_prompt = build_grounded_llm_prompt(
    user_query=grounding_test_query,
    recommendations=grounding_test_results
)


print("SYSTEM INSTRUCTIONS")
print("=" * 80)
print(
    test_prompt["system"]
)

print("\n" + "=" * 80)
print("USER PROMPT")
print("=" * 80)
print(
    test_prompt["user"]
)


# ------------------------------------------------------------
# Basic prompt integrity checks
# ------------------------------------------------------------

retrieved_titles = (
    grounding_test_results[
        "canonical_title"
    ]
    .tolist()
)

titles_present = all(
    title in test_prompt["user"]
    for title in retrieved_titles
)


print("\n" + "=" * 80)
print("PROMPT VALIDATION")
print("=" * 80)

print(
    "Retrieved books:",
    len(retrieved_titles)
)

print(
    "All retrieved titles present:",
    titles_present
)

print(
    "User query present:",
    grounding_test_query
    in test_prompt["user"]
)

print(
    "Grounding restriction present:",
    "Recommend only books contained"
    in test_prompt["system"]
)

print(
    "Ranking preservation present:",
    "Preserve the recommendation ranking"
    in test_prompt["system"]
)

print(
    "Anti-fabrication rule present:",
    "Do not invent authors"
    in test_prompt["system"]
)

print(
    "Similarity-score warning present:",
    "not a quality score"
    in test_prompt["system"]
)

print(
    "\nSystem prompt characters:",
    len(test_prompt["system"])
)

print(
    "User prompt characters:",
    len(test_prompt["user"])
)

SYSTEM INSTRUCTIONS
You are the explanation layer of a leadership and management
book recommendation system.

The books have already been selected and ranked by a
TF-IDF cosine-similarity retrieval model.

Your role is to explain the retrieved recommendations clearly
and accurately.

STRICT GROUNDING RULES:

1. Recommend only books contained in the RETRIEVED BOOK EVIDENCE.
2. Preserve the recommendation ranking supplied by the retrieval model.
3. Do not introduce additional book titles.
4. Do not invent authors, descriptions, subjects, ratings,
   publication years, or other metadata.
5. If metadata says "Not available", do not infer the missing value.
6. Base book-specific explanations only on the supplied evidence.
7. A similarity score represents textual similarity to the user's
   request. It is not a quality score, predicted rating, probability
   of satisfaction, or expert evaluation.
8. Topic-cluster labels provide supplementary context only.
9. Do not claim that the retrieval m

## 9. LLM API Integration

The retrieval and grounding pipeline has now been validated independently of any external language model.

The next stage connects the grounded prompt to an LLM.

The LLM is used only as the **natural-language explanation layer**. It does not select, rank, or independently generate book recommendations.

The complete pipeline becomes:

**User Query**

→ **Validated Enriched TF-IDF Vectorizer**

→ **Cosine-Similarity Retrieval**

→ **Top-N Books**

→ **Structured Grounding Context**

→ **Grounded LLM Prompt**

→ **Natural-Language Recommendation Explanation**

### API Security

The API key must not be:

- written directly into the notebook;
- stored in source code;
- printed in notebook output;
- committed to GitHub.

During notebook development, the key will be entered securely at runtime.

For later Streamlit deployment, the credential should be stored using the application's secret-management mechanism rather than embedded in the source code.

### Reproducibility

The recommendation ranking remains reproducible independently of the LLM because retrieval is performed before generation.

LLM wording may vary between requests, but the underlying retrieved books, similarity scores, and dataset evidence remain controlled by the recommendation pipeline.

In [14]:
# ============================================================
# INSTALL OPENAI SDK INTO CURRENT PROJECT ENVIRONMENT
# ============================================================

import sys
import subprocess

print("Python executable:")
print(sys.executable)

print("\nInstalling OpenAI SDK...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "openai"
])

print("\nInstallation completed.")

Python executable:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/.venv/bin/python

Installing OpenAI SDK...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.5 MB/s  0:00:007.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 5.9 MB/s  0:00:00m 6.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [openai];5;237m━━━━ 8/9 [openai]

Installation completed.


In [15]:
# ============================================================
# VERIFY OPENAI SDK
# ============================================================

import openai

print("OPENAI SDK VALIDATION")
print("=" * 80)

print(
    "OpenAI SDK version:",
    openai.__version__
)

print(
    "Installation successful:",
    True
)

OPENAI SDK VALIDATION
OpenAI SDK version: 3.19.0
Installation successful: True


### 9.1 Secure API Authentication

The OpenAI API requires an API key for authentication.

For notebook development, the key is entered at runtime using Python's `getpass` function.

This approach prevents the credential from:

- appearing directly in the notebook source code;
- being displayed during normal input;
- being accidentally committed to the project repository.

The API key is used only to initialize the OpenAI client for the current Python session.

For the later Streamlit application and cloud deployment, the API key should be stored using environment variables or the deployment platform's secret-management system rather than being embedded in application code.

In [17]:
# ============================================================
# SECURE OPENAI API AUTHENTICATION
# ============================================================

from getpass import getpass
from openai import OpenAI


api_key = getpass(
    "Enter your OpenAI API key: "
)

client = OpenAI(
    api_key=api_key
)


print("OPENAI CLIENT CONFIGURATION")
print("=" * 80)

print(
    "API key entered:",
    bool(api_key)
)

print(
    "OpenAI client created:",
    client is not None
)

print(
    "API key displayed:",
    False
)

Enter your OpenAI API key:  ········


OPENAI CLIENT CONFIGURATION
API key entered: True
OpenAI client created: True
API key displayed: False


## 9. Provider-Independent LLM Interface

The retrieval and grounding components of the recommendation system do not depend on a specific LLM provider.

A live external API is therefore treated as an optional generation component rather than a dependency of the recommendation engine.

The architecture is designed as:

**User Query**

→ **TF-IDF Retrieval**

→ **Grounded Book Evidence**

→ **Controlled Prompt**

→ **LLM Provider (Optional)**

→ **Natural-Language Explanation**

This modular design allows the recommendation and grounding pipeline to be developed, tested, and evaluated independently of API credentials.

During the current development stage, the system operates in **offline grounding mode**. The generated prompt can be inspected and validated without sending it to an external model.

A live LLM provider can subsequently be connected without retraining or modifying the underlying recommendation model.

In [18]:
# ============================================================
# PROVIDER-INDEPENDENT LLM CONFIGURATION
# ============================================================

LLM_CONFIG = {
    "provider": None,
    "model": None,
    "live_api_enabled": False,
    "mode": "offline_grounding"
}

print("LLM CONFIGURATION")
print("=" * 80)

for key, value in LLM_CONFIG.items():
    print(f"{key:<20}: {value}")

print(
    "\nRecommendation engine operational:",
    True
)

print(
    "Grounding pipeline operational:",
    True
)

print(
    "Live LLM generation required:",
    False
)

LLM CONFIGURATION
provider            : None
model               : None
live_api_enabled    : False
mode                : offline_grounding

Recommendation engine operational: True
Grounding pipeline operational: True
Live LLM generation required: False


## 10. Offline Grounded Recommendation Mode

Because a live LLM API is not currently enabled, the system provides an offline presentation layer for the retrieved recommendations.

This component is **not an LLM simulation**.

It does not generate new semantic interpretations or imitate a language model. Instead, it converts the retrieved and validated book metadata into a structured user-facing response.

This distinction is important for methodological transparency.

### Offline Workflow

**Natural-Language User Query**

→ **Validated TF-IDF Transformation**

→ **Cosine-Similarity Retrieval**

→ **Top-N Books**

→ **Structured Grounding Metadata**

→ **Deterministic Recommendation Presentation**

The offline mode preserves:

- recommendation ranking;
- book titles and authors;
- retrieval similarity scores;
- available subjects;
- topic-cluster information;
- missing-data transparency.

When an external LLM provider is later connected, the deterministic presentation layer can be replaced by the grounded LLM generation function without modifying the underlying recommendation engine.

In [19]:
# ============================================================
# OFFLINE GROUNDED RECOMMENDATION PRESENTER
# ============================================================

def format_authors(value):
    """
    Convert stored author metadata into readable text
    without inventing missing information.
    """

    if pd.isna(value):
        return "Author information not available"

    text = str(value).strip()

    if text in ["", "[]"]:
        return "Author information not available"

    # Clean simple list-style formatting
    text = (
        text
        .replace("[", "")
        .replace("]", "")
        .replace("'", "")
    )

    return text


def format_optional_metadata(value):
    """
    Represent unavailable metadata consistently.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text in ["", "[]", "Not available"]:
        return None

    return text


def present_offline_recommendations(
    user_query,
    recommendations
):
    """
    Produce a deterministic, grounded recommendation response.

    This is NOT an LLM-generated response.
    """

    if recommendations.empty:

        return (
            "No recommendations could be retrieved "
            "for this request."
        )


    lines = []

    lines.append(
        "Your request focuses on:"
    )

    lines.append(
        f'"{user_query}"'
    )

    lines.append("")

    lines.append(
        "The recommendation system retrieved the "
        "following books:"
    )

    lines.append("")


    for _, row in recommendations.iterrows():

        rank = int(
            row["recommendation_rank"]
        )

        title = (
            row["canonical_title"]
        )

        authors = format_authors(
            row["authors"]
        )

        similarity = (
            row["similarity_score"]
        )

        cluster = format_optional_metadata(
            row["cluster_label"]
        )

        subjects = format_optional_metadata(
            row["subjects"]
        )


        lines.append(
            f"{rank}. {title}"
        )

        lines.append(
            f"   Author(s): {authors}"
        )

        lines.append(
            f"   Retrieval similarity: "
            f"{similarity:.4f}"
        )


        if cluster:

            lines.append(
                f"   Topic: {cluster}"
            )


        if subjects:

            lines.append(
                f"   Available subjects: "
                f"{subjects}"
            )


        lines.append("")


    lines.append(
        "Note: These recommendations are ranked using "
        "textual similarity between your request and the "
        "available metadata in the project catalog. "
        "The similarity score is not a book-quality rating."
    )


    return "\n".join(
        lines
    )

In [20]:
# ============================================================
# END-TO-END OFFLINE RECOMMENDATION TEST
# ============================================================

offline_test_query = (
    "I want to become a better leader by improving "
    "decision making, communication, team leadership, "
    "and strategic thinking."
)


# Step 1 — Retrieve
offline_results = retrieve_books_from_query(
    query=offline_test_query,
    top_n=5
)


# Step 2 — Build LLM-ready prompt
offline_prompt = build_grounded_llm_prompt(
    user_query=offline_test_query,
    recommendations=offline_results
)


# Step 3 — Produce deterministic offline presentation
offline_response = present_offline_recommendations(
    user_query=offline_test_query,
    recommendations=offline_results
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("END-TO-END OFFLINE TEST")
print("=" * 80)

print(
    "Query:",
    offline_test_query
)

print(
    "\nRecommendations retrieved:",
    len(offline_results)
)

print(
    "Unique books:",
    offline_results["book_id"].nunique()
)

print(
    "LLM-ready prompt created:",
    bool(offline_prompt["user"])
)

print(
    "Live API used:",
    False
)


print("\n" + "=" * 80)
print("USER-FACING OFFLINE RESPONSE")
print("=" * 80)

print(
    offline_response
)

END-TO-END OFFLINE TEST
Query: I want to become a better leader by improving decision making, communication, team leadership, and strategic thinking.

Recommendations retrieved: 5
Unique books: 5
LLM-ready prompt created: True
Live API used: False

USER-FACING OFFLINE RESPONSE
Your request focuses on:
"I want to become a better leader by improving decision making, communication, team leadership, and strategic thinking."

The recommendation system retrieved the following books:

1. Team Leadership
   Author(s): Drikus Kriek
   Retrieval similarity: 0.3440
   Topic: Team Leadership

2. Strategic Team Leadership
   Author(s): Peter Saul
   Retrieval similarity: 0.3349
   Topic: Team Leadership

3. The Six Disciplines of Strategic Thinking
   Author(s): Michael D. Watkins
   Retrieval similarity: 0.3290
   Topic: CEO and Executive Transformation

4. Team leadership
   Author(s): John Apps
   Retrieval similarity: 0.2977
   Topic: Team Leadership

5. If You Want Something Done
   Author(s):

## 11. Retrieval Robustness and Edge-Case Testing

Before finalizing the LLM integration pipeline, the natural-language retrieval function is evaluated against several edge cases.

The objective is not to measure supervised recommendation accuracy. Instead, these tests verify that the system behaves predictably when presented with different types of user input.

Four cases are evaluated:

1. **Empty query** — verifies that missing user input does not generate arbitrary recommendations.
2. **Out-of-vocabulary query** — verifies that text with no recognized TF-IDF features does not produce artificial results.
3. **Focused query** — evaluates retrieval for a narrowly defined leadership topic.
4. **Broad query** — evaluates whether a general management request still produces valid recommendations.

A robust retrieval system should return no recommendations when there is insufficient textual evidence rather than padding the output with unrelated books.

In [21]:
# ============================================================
# RETRIEVAL ROBUSTNESS TESTS
# ============================================================

edge_case_queries = {
    "Empty Query": "",
    
    "Out of Vocabulary": (
        "qxzvplm jjjkkk zzzqqq"
    ),
    
    "Focused Query": (
        "negotiation and conflict resolution"
    ),
    
    "Broad Query": (
        "leadership management"
    )
}


edge_case_summary = []


for test_name, query_text in edge_case_queries.items():

    results = retrieve_books_from_query(
        query=query_text,
        top_n=5
    )

    if results.empty:

        edge_case_summary.append({
            "test": test_name,
            "query": query_text,
            "recommendations": 0,
            "unique_books": 0,
            "all_positive_similarity": None,
            "top_result": None
        })

    else:

        edge_case_summary.append({
            "test": test_name,
            "query": query_text,
            "recommendations": len(results),
            "unique_books": results["book_id"].nunique(),
            "all_positive_similarity": bool(
                (
                    results["similarity_score"]
                    > 0
                ).all()
            ),
            "top_result": results.iloc[0][
                "canonical_title"
            ]
        })


edge_case_summary_df = pd.DataFrame(
    edge_case_summary
)


print("RETRIEVAL ROBUSTNESS TESTS")
print("=" * 100)

display(
    edge_case_summary_df
)


# ------------------------------------------------------------
# Show recommendations for successful queries
# ------------------------------------------------------------

for test_name in [
    "Focused Query",
    "Broad Query"
]:

    query_text = edge_case_queries[
        test_name
    ]

    results = retrieve_books_from_query(
        query=query_text,
        top_n=5
    )

    print(
        "\n" + "=" * 100
    )

    print(
        test_name.upper()
    )

    print(
        "Query:",
        query_text
    )

    print(
        "=" * 100
    )

    if not results.empty:

        display(
            results[
                [
                    "recommendation_rank",
                    "canonical_title",
                    "authors",
                    "similarity_score",
                    "cluster_label"
                ]
            ].style.format({
                "similarity_score": "{:.4f}"
            })
        )

The query contains no vocabulary recognized by the fitted TF-IDF model.
RETRIEVAL ROBUSTNESS TESTS


,test,query,recommendations,unique_books,all_positive_similarity,top_result
0,Empty Query,,0,0,None,NaN
1,Out of Vocabulary,qxzvplm jjjkkk zzzqqq,0,0,None,NaN
2,Focused Query,negotiation and conflict resolution,5,5,True,The Seven Tensions of Negotiation
3,Broad Query,leadership management,5,5,True,Leadership Development Studies



FOCUSED QUERY
Query: negotiation and conflict resolution


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,The Seven Tensions of Negotiation,['Cash Nickerson'],0.3166,Organizational Culture and Trust
1,2,The Elements of Negotiation,['Keld Jensen'],0.3104,Organizational Culture and Trust
2,3,From Conflict to Convergence,['Robert Fersh and Mariah Levison'],0.2311,Organizational Culture and Trust
3,4,From Conflict to Courage,['Marlene Chism'],0.2061,Organizational Culture and Trust
4,5,Conflict Resilience,['Robert Bordone and Joel Salinas M.D.'],0.2043,Organizational Culture and Trust



BROAD QUERY
Query: leadership management


,recommendation_rank,canonical_title,authors,similarity_score,cluster_label
0,1,Leadership Development Studies,"['Monika Byrd', 'Susan Edwards']",0.5076,Leadership Development
1,2,The Soil of Leadership,['Britt Yamamoto'],0.4216,Transformational Leadership
2,3,Kepemimpinan =,['Karjadi M.'],0.4216,Transformational Leadership
3,4,Leadership Unblocked,['Muriel M. Wilkins'],0.4216,Transformational Leadership
4,5,Flow Leadership,['Gaelle Devins'],0.4216,Transformational Leadership


## 12. Save LLM Integration Artifacts

The retrieval-grounded LLM integration pipeline has now been validated.

Although a live external LLM API was not used, the notebook implements and validates the complete provider-independent architecture required to connect a language model later.

The saved artifacts preserve:

1. the structured LLM grounding catalog;
2. the natural-language retrieval diagnostic results;
3. the retrieval robustness tests;
4. the validated system instructions used for grounded generation;
5. the provider-independent LLM configuration.

### Current Operating Mode

The system currently operates in:

**Offline Grounding Mode**

This means that:

- natural-language retrieval is operational;
- TF-IDF recommendation ranking is operational;
- structured grounding is operational;
- LLM-ready prompt generation is operational;
- deterministic offline presentation is operational;
- live external LLM inference is not enabled.

No simulated LLM output is presented as genuine generative-model output.

This preserves methodological transparency while allowing a live provider to be connected later without changing the recommendation architecture.

In [22]:
# ============================================================
# SAVE LLM INTEGRATION ARTIFACTS
# ============================================================

LLM_GROUNDING_PATH = (
    PROCESSED_DIR
    / "llm_grounding_catalog.csv"
)

LLM_DIAGNOSTIC_PATH = (
    PROCESSED_DIR
    / "llm_retrieval_diagnostic.csv"
)

LLM_ROBUSTNESS_PATH = (
    PROCESSED_DIR
    / "llm_retrieval_robustness.csv"
)

LLM_SYSTEM_PROMPT_PATH = (
    MODELS_DIR
    / "llm_system_instructions.txt"
)

LLM_CONFIG_PATH = (
    MODELS_DIR
    / "llm_config.json"
)


# ------------------------------------------------------------
# 1. Grounding catalog
# ------------------------------------------------------------

grounding_catalog.to_csv(
    LLM_GROUNDING_PATH,
    index=False
)


# ------------------------------------------------------------
# 2. Multi-query diagnostic
# ------------------------------------------------------------

diagnostic_summary_df.to_csv(
    LLM_DIAGNOSTIC_PATH,
    index=False
)


# ------------------------------------------------------------
# 3. Robustness evaluation
# ------------------------------------------------------------

edge_case_summary_df.to_csv(
    LLM_ROBUSTNESS_PATH,
    index=False
)


# ------------------------------------------------------------
# 4. Grounded system instructions
# ------------------------------------------------------------

LLM_SYSTEM_PROMPT_PATH.write_text(
    LLM_SYSTEM_INSTRUCTIONS,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. Provider-independent configuration
# ------------------------------------------------------------

with open(
    LLM_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        LLM_CONFIG,
        file,
        indent=4
    )


# ------------------------------------------------------------
# Reload for validation
# ------------------------------------------------------------

grounding_reload = pd.read_csv(
    LLM_GROUNDING_PATH
)

diagnostic_reload = pd.read_csv(
    LLM_DIAGNOSTIC_PATH
)

robustness_reload = pd.read_csv(
    LLM_ROBUSTNESS_PATH
)

system_prompt_reload = (
    LLM_SYSTEM_PROMPT_PATH
    .read_text(
        encoding="utf-8"
    )
)

with open(
    LLM_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    config_reload = json.load(
        file
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("LLM INTEGRATION ARTIFACT VALIDATION")
print("=" * 80)

print(
    "Grounding catalog:",
    LLM_GROUNDING_PATH.exists()
)

print(
    "Retrieval diagnostic:",
    LLM_DIAGNOSTIC_PATH.exists()
)

print(
    "Robustness evaluation:",
    LLM_ROBUSTNESS_PATH.exists()
)

print(
    "System instructions:",
    LLM_SYSTEM_PROMPT_PATH.exists()
)

print(
    "LLM configuration:",
    LLM_CONFIG_PATH.exists()
)


print("\nGROUNDING VALIDATION")
print("=" * 80)

print(
    "Grounding rows:",
    len(grounding_reload)
)

print(
    "Grounding columns:",
    grounding_reload.shape[1]
)

print(
    "Unique book IDs:",
    grounding_reload[
        "book_id"
    ].nunique()
)

print(
    "Expected books:",
    len(grounding_reload) == 2067
)


print("\nEVALUATION VALIDATION")
print("=" * 80)

print(
    "Diagnostic queries:",
    len(diagnostic_reload)
)

print(
    "Robustness tests:",
    len(robustness_reload)
)


print("\nPROMPT VALIDATION")
print("=" * 80)

print(
    "System prompt preserved:",
    system_prompt_reload
    == LLM_SYSTEM_INSTRUCTIONS
)

print(
    "Grounding rule preserved:",
    "Recommend only books contained"
    in system_prompt_reload
)


print("\nCONFIGURATION")
print("=" * 80)

print(
    "Provider:",
    config_reload["provider"]
)

print(
    "Live API enabled:",
    config_reload[
        "live_api_enabled"
    ]
)

print(
    "Mode:",
    config_reload["mode"]
)


print("\nSaved artifacts:")
print(LLM_GROUNDING_PATH)
print(LLM_DIAGNOSTIC_PATH)
print(LLM_ROBUSTNESS_PATH)
print(LLM_SYSTEM_PROMPT_PATH)
print(LLM_CONFIG_PATH)

LLM INTEGRATION ARTIFACT VALIDATION
Grounding catalog: True
Retrieval diagnostic: True
Robustness evaluation: True
System instructions: True
LLM configuration: True

GROUNDING VALIDATION
Grounding rows: 2067
Grounding columns: 19
Unique book IDs: 2067
Expected books: True

EVALUATION VALIDATION
Diagnostic queries: 6
Robustness tests: 4

PROMPT VALIDATION
System prompt preserved: True
Grounding rule preserved: True

CONFIGURATION
Provider: None
Live API enabled: False
Mode: offline_grounding

Saved artifacts:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/llm_grounding_catalog.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/llm_retrieval_diagnostic.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/llm_retrieval_robustness.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book

# Notebook 15 — LLM Integration: Final Summary

## Objective

This notebook extended the Leadership and Management Book Recommendation System with a provider-independent architecture for Large Language Model integration.

The objective was not to replace the validated recommendation model with an LLM.

Instead, the LLM layer was designed to support:

- natural-language user requests;
- grounded recommendation explanations;
- conversational presentation of retrieved books;
- transparent handling of unavailable metadata.

The underlying recommendation ranking remains controlled by the validated content-based recommendation system.

---

## Final Architecture

The resulting architecture is:

**Natural-Language User Request**

→ **Validated Enriched TF-IDF Vectorizer**

→ **Cosine-Similarity Retrieval**

→ **Ranked Top-N Books**

→ **Structured Book Grounding**

→ **Controlled LLM Prompt**

→ **LLM Explanation (Optional)**

or, when no live LLM provider is configured:

→ **Deterministic Offline Presentation**

This separates recommendation retrieval from natural-language generation.

---

## Recommendation Representation

The notebook successfully reloaded the validated enriched TF-IDF representation:

- **2,067 books**
- **5,130 TF-IDF features**
- **27 zero-vector books**
- **2,040 recommendation-eligible books**

The existing vectorizer vocabulary and TF-IDF matrix remained fully aligned.

No new TF-IDF model was trained.

---

## Structured Grounding Catalog

A dedicated grounding catalog was constructed for all:

**2,067 books**

with **19 structured fields**.

The grounding catalog combines the authoritative TF-IDF matrix-row mapping with structured metadata and available topic-cluster information.

Important metadata coverage included:

- Authors: **91.15%**
- Subjects: **91.15%**
- Topic clusters: **91.15%**
- Cluster labels: **91.15%**
- Cover images: **82.49%**
- Edition count: **45.96%**
- First publication year: **45.82%**
- Observed publication year: **45.19%**
- Ratings: **13.64%**
- Descriptions: **9.29%**

The relatively low description and rating coverage is explicitly preserved rather than filled using inferred or fabricated information.

---

## Natural-Language Retrieval

A new query-retrieval function allows users to describe their leadership or management needs directly.

Example:

> "I am leading organizational change and need help managing transformation, employee resistance, culture, and change leadership."

The query is transformed using the existing fitted enriched TF-IDF vectorizer and compared against the book catalog using cosine similarity.

Only books with positive similarity and valid TF-IDF representations are returned.

This enables recommendation without requiring the user to begin with an existing book title.

---

## Multi-Query Diagnostic Evaluation

Natural-language retrieval was evaluated across six leadership and management scenarios:

1. New Manager
2. Conflict Management
3. Strategy
4. Organizational Change
5. Entrepreneurship
6. Employee Motivation

All six tests returned:

- **10 recommendations**
- **10 unique books**

The retrieved topic distributions changed according to user intent.

Examples included:

- New Manager → Organizational Culture and Trust
- Strategy → Decision Making / Strategic Management
- Organizational Change → Change Management
- Entrepreneurship → Entrepreneurial and Growth Mindset

This provides evidence that the retrieval model responds to different natural-language intents rather than returning a static recommendation set.

However, some broader or multi-concept queries produced less specific results, demonstrating the limitations of lexical TF-IDF retrieval.

---

## Retrieval Robustness

Four additional edge cases were evaluated.

### Empty Query

**0 recommendations**

The system does not generate arbitrary recommendations when no query is provided.

### Out-of-Vocabulary Query

**0 recommendations**

A query containing no recognized TF-IDF vocabulary does not produce artificial recommendations.

### Focused Query

Query:

> "negotiation and conflict resolution"

Result:

- **5 recommendations**
- **5 unique books**
- all similarities positive

The retrieved books were strongly concentrated around negotiation and conflict-related topics.

### Broad Query

Query:

> "leadership management"

Result:

- **5 recommendations**
- **5 unique books**
- all similarities positive

The recommendations were appropriately broader and more general.

These tests demonstrate that the retrieval function fails safely when insufficient textual evidence is available.

---

## Grounded LLM Prompt

A controlled prompt architecture was developed for future LLM integration.

The prompt requires the LLM to:

- recommend only retrieved books;
- preserve the retrieval ranking;
- avoid introducing additional titles;
- avoid fabricating unavailable metadata;
- distinguish textual similarity from book quality;
- use topic clusters only as supplementary context;
- acknowledge limited metadata when appropriate.

The prompt was validated to ensure that:

- all retrieved titles were included;
- the original user query was preserved;
- ranking-preservation instructions were present;
- anti-fabrication rules were present;
- similarity-score interpretation was explicitly constrained.

---

## Retrieval-Augmented Generation Design

The notebook therefore implements the core structure of a retrieval-augmented generation workflow:

**Retrieve → Ground → Generate**

The recommendation model determines **which books are retrieved**.

The structured catalog determines **what book-specific evidence is available**.

A future LLM determines only **how that evidence is communicated**.

This separation reduces the risk of allowing a generative model to independently invent or substitute book recommendations.

---

## Offline Grounding Mode

No live external LLM API credentials were configured during this experiment.

The current configuration is therefore:

- Provider: **None**
- Model: **None**
- Live API enabled: **False**
- Mode: **offline_grounding**

A deterministic offline recommendation presenter was implemented as a fallback.

This component is explicitly **not described as an LLM**.

It formats retrieved book evidence without performing generative inference.

---

## End-to-End Validation

The complete offline workflow was successfully tested:

**Natural-Language Query**

→ **TF-IDF Transformation**

→ **Cosine Retrieval**

→ **Top-N Recommendation**

→ **Structured Grounding**

→ **LLM-Ready Prompt**

→ **Offline User-Facing Presentation**

The end-to-end test successfully returned:

- **5 recommendations**
- **5 unique books**
- positive similarity scores
- grounded metadata
- a valid LLM-ready prompt

No live API request was required.

---

## Methodological Limitations

Several limitations should be considered.

### 1. Lexical Sensitivity

TF-IDF depends strongly on vocabulary overlap.

Distinctive phrases can dominate multi-concept queries.

### 2. Broad Queries

General requests such as "leadership management" provide limited discriminating information and can produce broader recommendations or tied similarity scores.

### 3. Metadata Heterogeneity

The catalog combines sources with different levels of metadata richness.

This can influence textual similarity and recommendation behavior.

### 4. Sparse Descriptions

Only **9.29%** of books contain descriptions.

LLM explanations therefore cannot consistently rely on book descriptions.

### 5. No Human Relevance Labels

The project does not contain explicit user relevance judgments.

The natural-language retrieval evaluation is therefore diagnostic rather than a supervised accuracy evaluation.

### 6. No Live LLM Evaluation

The grounding architecture and prompt were validated, but no live LLM inference was executed.

Consequently, this notebook does not make claims about:

- LLM response quality;
- hallucination rate;
- generation latency;
- API cost;
- provider-specific performance.

These should be evaluated if a live LLM provider is connected later.

---

## Saved Artifacts

### Grounding Catalog

`data/processed/llm_grounding_catalog.csv`

- 2,067 books
- 19 grounding fields

### Retrieval Diagnostic

`data/processed/llm_retrieval_diagnostic.csv`

Contains results from the six natural-language retrieval scenarios.

### Retrieval Robustness

`data/processed/llm_retrieval_robustness.csv`

Contains the four edge-case retrieval tests.

### LLM System Instructions

`models/llm_system_instructions.txt`

Contains the validated grounding and anti-fabrication instructions.

### LLM Configuration

`models/llm_config.json`

Records the provider-independent offline configuration.

---

## Final Model Decision

The LLM is not treated as the recommendation model.

The validated production recommendation architecture remains:

**Enriched TF-IDF → Cosine Similarity → Duplicate Suppression → Ranked Top-N Recommendations**

For natural-language requests, the extended architecture becomes:

**Natural-Language Query → Enriched TF-IDF → Cosine Similarity → Ranked Top-N → Grounded Context → Optional LLM Explanation**

The LLM layer therefore enhances interaction and explanation while leaving recommendation retrieval under the control of the validated content-based model.

---

## Final Status

**Notebook 15 — LLM Integration: COMPLETE**

The project now contains:

**NLP**

→ **Dimensionality Reduction**

→ **Clustering**

→ **Recommendation System**

→ **PyTorch Tensors**

→ **Neural Network**

→ **LLM-Ready Grounded Recommendation Architecture**

The next stage can proceed without modifying the validated recommendation model.